# Predicting EV Purchases: Features and Models

Notebook 2 of 3 for the Kaggle competition Playground Series S6E9 (predict `Will_Buy_EV`, scored by ROC AUC). This notebook builds the features, encodes income and commute values with the target inside each fold, tunes LightGBM with Optuna, and trains LightGBM, XGBoost and a logistic regression with 10-fold cross-validation. The code is the code of the submitted version, moved out of one notebook into three.

It reads the competition files (CC BY 4.0) and writes `own_predictions.npz` (the out-of-fold and test predictions of the three models, with the row ids), `lgb_params.json` (the tuned LightGBM parameters) and two figures to `figures/`. Notebook 3 reads `own_predictions.npz` and stacks the three models with public out-of-fold predictions.

## Setup

Import the libraries and set the main settings: the random seed, the number of folds and the tuning budget (20 trials or 600 seconds, whichever ends first). `SMOKE` samples a small part of the data and runs 2 trials for a quick check run on Kaggle; it is off in this version.

In [ ]:
import glob
import hashlib
import json
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import sklearn
import xgboost as xgb
from scipy.stats import norm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid")

SMOKE = False
SEED = 42
N_FOLDS = 10
N_TRIALS = 20
TUNE_TIMEOUT = 600
DEVICE = "cpu"
if SMOKE:
    N_TRIALS = 2
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)
print(f"smoke run: {SMOKE}")
print(f"lightgbm {lgb.__version__}, xgboost {xgb.__version__}, optuna {optuna.__version__}, scikit-learn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")

**Conclusion.** The smoke switch is off. The Kaggle image provides LightGBM 4.6.0, XGBoost 3.4.1, Optuna 5.0.0, scikit-learn 1.6.1, pandas 2.3.3 and NumPy 2.1.3, so nothing has to be installed.

## Load the data

Read the train, test and sample submission files and turn the target into 0 and 1.

In [ ]:
root = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0].rsplit("/", 1)[0]
train = pd.read_csv(f"{root}/train.csv")
test = pd.read_csv(f"{root}/test.csv")
sample = pd.read_csv(f"{root}/sample_submission.csv")
if SMOKE:
    train = train.sample(30_000, random_state=SEED).reset_index(drop=True)
    test = test.sample(10_000, random_state=SEED).reset_index(drop=True)

TARGET = "Will_Buy_EV"
y = (train[TARGET] == "Yes").astype(int).to_numpy()
print(train.shape, test.shape)
print(train.isna().sum().sum(), test.isna().sum().sum())
print(f"positive rate: {y.mean():.4f}")

**Conclusion.** These are the same 668,665 training rows and 286,571 test rows as in notebook 1, with no missing values and a purchase rate of 17.46%.

## Features and the recipe score

The data looks like a copy of a simple rule-based dataset. In that rule, a person buys when a score is high enough:

`score = 1.2 * income / 100000 + 0.6 * environmental concern + 2 * subsidy - 1 * medium range anxiety - 3 * high range anxiety`

Some random noise is added, and the person buys if the score is above 5.5. This gives `P(buy) = Phi(score - 5.5)`, where Phi is the normal cumulative distribution. The score is added as a feature, and its logit is also the starting point (base margin) of the boosted models, so the trees only have to learn what the rule misses.

Other features: a worry score about charging, the total number of charging stations, income times subsidy, and rounded versions of income and commute that serve as keys for the target encoding. The text categories become integer codes, and frequency columns count how often each income and commute value appears in train and test together. The cell prints the AUC of the recipe score on its own.

In [ ]:
RANGE_MAP = {"Low": 0, "Medium": 1, "High": 2}
CATS = ["Gender", "City_Type", "Current_Car_Type"]
TE_KEYS = ["income_key", "income_10", "income_100", "income_1000", "commute_10", "age_income", "income_commute"]
DROP_KEYS = ["age_income", "income_commute", "income_10", "income_100", "income_1000", "commute_10"]


def recipe_score(df):
    anxiety = df.Range_Anxiety_Level.map(RANGE_MAP)
    return (1.2 * df.Annual_Income_USD / 1e5 + 0.6 * df.Environmental_Concern_Level
            + 2.0 * (df.Subsidy_Available == "Yes") - np.select([anxiety == 2, anxiety == 1], [3.0, 1.0], 0.0))


def to_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def build_features(df):
    out = df.drop(columns=["id", TARGET], errors="ignore").copy()
    home = (out.Home_Charging_Possible == "Yes").astype(int)
    stations = out.Charging_Stations_Near_Home + out.Charging_Stations_Near_Work

    out["recipe_score"] = recipe_score(out)
    out["worry_score"] = out.Daily_Commute_km - 5 * stations - 150 * home
    out["total_stations"] = stations
    out["income_x_subsidy"] = out.Annual_Income_USD / 1e5 * (out.Subsidy_Available == "Yes")

    out["Subsidy_Available"] = (out.Subsidy_Available == "Yes").astype(int)
    out["Home_Charging_Possible"] = home
    out["Range_Anxiety_Level"] = out.Range_Anxiety_Level.map(RANGE_MAP)
    for col in CATS:
        out[col] = out[col].astype("category").cat.codes

    income = out.Annual_Income_USD.astype(int)
    out["income_key"] = income
    out["income_10"] = income // 10
    out["income_100"] = income // 100
    out["income_1000"] = income // 1000
    out["commute_10"] = (out.Daily_Commute_km * 10).round().astype(int)
    out["age_income"] = out.Age * 1_000_000 + income
    out["income_commute"] = income * 10_000 + out.commute_10
    return out


full = build_features(pd.concat([train, test], ignore_index=True))
for col in ["income_key", "age_income", "income_commute", "commute_10"]:
    full[f"{col}_freq"] = full[col].map(full[col].value_counts(normalize=True))

X_all = full.iloc[: len(train)].reset_index(drop=True)
X_test = full.iloc[len(train):].reset_index(drop=True)
margin_all = to_logit(norm.cdf(X_all.recipe_score - 5.5))
margin_test = to_logit(norm.cdf(X_test.recipe_score - 5.5))
recipe_auc = roc_auc_score(y, X_all.recipe_score)
print(X_all.shape, X_test.shape)
print(f"recipe score AUC: {recipe_auc:.4f}")

**Conclusion.** Train and test both have 28 columns after feature building. The recipe score on its own reaches an AUC of 0.9377 with no model at all, which confirms that the data follows the rule closely; the models below have to find the remaining signal on top of it.

## Target encoding inside each fold

Target encoding replaces a value (for example an exact income) by the purchase rate seen for that value in the training rows. It is powerful, but it leaks the answer if a row can see its own label. To avoid this:

- The encoding for the training rows of a fold is built with an inner 5-fold split, so a row never sees its own label.
- The validation and test rows are encoded with the statistics of the full training part of the fold.

A smoothing value of 1 pulls rare values only slightly towards the overall rate. Income and commute also get a kernel-smoothed rate, which averages the purchase rate of nearby values with Gaussian weights, so a value with few rows can borrow information from its neighbours. The cell builds all 10 folds and prints the number of feature columns and the time taken.

In [ ]:
start = time.time()
KERNELS = {"income_key": 0.02, "commute_10": 0.01}


def target_encode(fit_x, fit_y, apply_x, smoothing=1.0):
    prior = fit_y.mean()
    encoded = {}
    for key in TE_KEYS:
        stats = pd.DataFrame({"k": fit_x[key].to_numpy(), "y": fit_y}).groupby("k")["y"].agg(["sum", "count"])
        mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
        encoded[f"te_{key}"] = apply_x[key].map(mapping).fillna(prior).to_numpy()
    for key, rate in KERNELS.items():
        lo = min(fit_x[key].min(), apply_x[key].min())
        size = int(max(fit_x[key].max(), apply_x[key].max()) - lo + 1)
        idx = fit_x[key].to_numpy() - lo
        half = int(np.ceil(np.sqrt(9 / rate)))
        weights = np.exp(-rate * np.arange(-half, half + 1) ** 2)
        num = np.convolve(np.bincount(idx, weights=fit_y, minlength=size), weights, mode="same")
        den = np.convolve(np.bincount(idx, minlength=size).astype(float), weights, mode="same")
        pos = apply_x[key].to_numpy() - lo
        encoded[f"kernel_{key}"] = (num[pos] + 5 * prior) / (den[pos] + 5)
    return pd.DataFrame(encoded, index=apply_x.index)


def finalize(x, te):
    return pd.concat([x, te], axis=1).drop(columns=DROP_KEYS).astype("float32")


def make_fold(train_idx, valid_idx):
    fit_x, fit_y = X_all.iloc[train_idx], y[train_idx]
    valid_x = X_all.iloc[valid_idx]
    inner = StratifiedKFold(5, shuffle=True, random_state=SEED + 1)
    fit_te = pd.concat([target_encode(fit_x.iloc[a], fit_y[a], fit_x.iloc[b]) for a, b in inner.split(fit_x, fit_y)]).loc[fit_x.index]
    return {
        "valid_idx": valid_idx,
        "X_train": finalize(fit_x, fit_te),
        "y_train": fit_y,
        "m_train": margin_all[train_idx],
        "X_valid": finalize(valid_x, target_encode(fit_x, fit_y, valid_x)),
        "y_valid": y[valid_idx],
        "m_valid": margin_all[valid_idx],
        "X_test": finalize(X_test, target_encode(fit_x, fit_y, X_test)),
    }


skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
folds = [make_fold(a, b) for a, b in skf.split(X_all, y)]
features = list(folds[0]["X_train"].columns)
print(f"folds: {len(folds)}, feature columns per fold: {len(features)}")
print(f"target encoding elapsed: {time.time() - start:.0f} seconds")

**Conclusion.** All 10 folds now have their own leak-free feature tables with 31 columns each, built in 31 seconds. Every model below uses exactly these folds, so their out-of-fold predictions can be compared and stacked fairly.

## Tuning LightGBM with Optuna

Optuna tries parameter sets with its TPE sampler (seed 42) and keeps the one with the best AUC on the first fold, to save time. The search stops after 20 trials or 600 seconds, whichever comes first, as in the submitted version. Each trial starts from the recipe margin and uses early stopping, so the number of trees is chosen automatically. The cell prints the best AUC, the number of trials that finished and the best parameters, and saves the optimisation history.

In [ ]:
def lgb_objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.05,
        "n_estimators": 4000,
        "num_leaves": trial.suggest_int("num_leaves", 6, 48, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 400, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 50, log=True),
        "max_bin": trial.suggest_categorical("max_bin", [255, 511, 1023]),
        "random_state": SEED,
        "verbose": -1,
    }
    f = folds[0]
    model = lgb.LGBMClassifier(**params)
    model.fit(f["X_train"], f["y_train"], init_score=f["m_train"], eval_set=[(f["X_valid"], f["y_valid"])],
              eval_init_score=[f["m_valid"]], callbacks=[lgb.early_stopping(100, verbose=False)])
    return roc_auc_score(f["y_valid"], model.predict(f["X_valid"], raw_score=True) + f["m_valid"])


start = time.time()
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(lgb_objective, n_trials=N_TRIALS, timeout=TUNE_TIMEOUT)
print(f"best fold-0 AUC {study.best_value:.5f}")
print(f"trials completed: {len(study.trials)} of {N_TRIALS}")
print(f"best parameters: {study.best_params}")
print(f"tuning elapsed: {time.time() - start:.0f} seconds")
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.savefig(FIGURES / "05_optuna_history.png", dpi=110, bbox_inches="tight")
plt.show()

**Conclusion.** The 600-second limit ended the search after 18 of the 20 trials (643 seconds in all, because the running trial is allowed to finish). The best trial reaches an AUC of 0.94512 on fold 0, and all 18 trials lie within about 0.0001 of each other (figure 05), so the result is not sensitive to the exact parameters. The best trial uses shallow trees (depth 3, at most 36 leaves) with at least 241 rows per leaf, which suits data that follows a simple rule plus noise: deep trees would mostly learn the noise. These parameters are reused on all folds with a lower learning rate.

## Cross-validation helper

One function trains a model on each fold and collects two things: the out-of-fold predictions (used to score the model and to build the stacker in notebook 3) and the test predictions averaged over the 10 folds. If several seeds are given, the results of all seeds are averaged. In an earlier version, bagging LightGBM and XGBoost over 3 seeds changed the out-of-fold AUC by less than 0.00002, so one seed is used. The cell also prints the size and purchase rate of every validation fold.

In [ ]:
def run_cv(name, fit_predict, seeds=(SEED,)):
    oof = np.zeros(len(train))
    pred = np.zeros(len(test))
    seed_scores = []
    for seed in seeds:
        seed_oof = np.zeros(len(train))
        for f in folds:
            valid_pred, test_pred = fit_predict(f, seed)
            seed_oof[f["valid_idx"]] = valid_pred
            pred += test_pred / (N_FOLDS * len(seeds))
        oof += seed_oof / len(seeds)
        seed_scores.append(roc_auc_score(y, seed_oof))
        print(f"{name} seed {seed}: OOF AUC {seed_scores[-1]:.5f}")
    bagged = roc_auc_score(y, oof)
    print(f"{name} bagged over {len(seeds)} seed(s): OOF AUC {bagged:.5f}")
    return oof, pred, seed_scores, bagged


for number, f in enumerate(folds):
    print(f"fold {number}: validation rows {len(f['valid_idx'])}, purchase rate {f['y_valid'].mean():.4f}")

**Conclusion.** Every validation fold holds 66,866 or 66,867 rows with a purchase rate of 0.1746, the same as the whole train set, so the stratified split gives each fold the same class balance.

## LightGBM

Train LightGBM on the 10 folds with the tuned parameters, a lower learning rate (0.03), up to 12,000 trees with early stopping after 300 rounds without improvement, and the recipe margin as the starting point.

In [ ]:
LGB_PARAMS = {**study.best_params, "objective": "binary", "metric": "auc", "learning_rate": 0.03,
              "n_estimators": 12000, "subsample_freq": 1, "verbose": -1}


def fit_lgb(f, seed):
    model = lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed)
    model.fit(f["X_train"], f["y_train"], init_score=f["m_train"], eval_set=[(f["X_valid"], f["y_valid"])],
              eval_init_score=[f["m_valid"]], callbacks=[lgb.early_stopping(300, verbose=False)])
    return (model.predict(f["X_valid"], raw_score=True) + f["m_valid"],
            model.predict(f["X_test"], raw_score=True) + margin_test)


start = time.time()
oof_lgb, test_lgb, scores_lgb, bag_lgb = run_cv("LightGBM", fit_lgb)
print(f"LightGBM elapsed: {time.time() - start:.0f} seconds")

**Conclusion.** LightGBM reaches an out-of-fold AUC of 0.94617, 0.0085 above the recipe score alone. The 10 folds took 1,115 seconds, the longest step of the notebook.

## XGBoost

XGBoost is a similar boosted tree model with different tree growing rules. Here it grows small trees leaf by leaf (at most 16 leaves and depth 4), again starting from the recipe margin. Its parameters are fixed, not tuned in this notebook.

In [ ]:
XGB_PARAMS = dict(n_estimators=10000, learning_rate=0.03, max_depth=4, max_leaves=16, grow_policy="lossguide",
                  min_child_weight=4.5, subsample=0.75, colsample_bytree=0.6, reg_alpha=0.75, reg_lambda=0.6,
                  gamma=1.0, max_bin=1024, tree_method="hist", device=DEVICE, eval_metric="auc",
                  early_stopping_rounds=300)


def fit_xgb(f, seed):
    model = xgb.XGBClassifier(**XGB_PARAMS, random_state=seed)
    model.fit(f["X_train"], f["y_train"], base_margin=f["m_train"], eval_set=[(f["X_valid"], f["y_valid"])],
              base_margin_eval_set=[f["m_valid"]], verbose=False)
    return (model.predict(f["X_valid"], base_margin=f["m_valid"], output_margin=True),
            model.predict(f["X_test"], base_margin=margin_test, output_margin=True))


start = time.time()
oof_xgb, test_xgb, scores_xgb, bag_xgb = run_cv("XGBoost", fit_xgb)
print(f"XGBoost elapsed: {time.time() - start:.0f} seconds")

**Conclusion.** XGBoost reaches 0.94616, within 0.00001 of LightGBM, in 727 seconds. Two boosted tree models on the same features end up almost identical, which notebook 3 confirms with their rank correlation.

## Logistic regression

A linear model is a very different kind of model from trees, so it can fix different mistakes. The purchase rule is close to additive, so a regularised logistic regression over the recipe score, dummy columns, interactions with subsidy and the logit of every target encoding works well. The features are standardised before fitting.

In [ ]:
def linear_matrix(x):
    out = pd.DataFrame({
        "recipe": x.recipe_score - 5.5,
        "income": x.Annual_Income_USD / 1e5,
        "subsidy": x.Subsidy_Available,
        "home": x.Home_Charging_Possible,
        "worry": x.worry_score / 100,
        "spike": (x.Annual_Income_USD == 30000).astype("float32"),
        "cliff": (x.Annual_Income_USD >= 170537).astype("float32"),
        "dead": x.Annual_Income_USD.between(31004, 41970).astype("float32"),
        "income_subsidy": x.income_x_subsidy,
        "income_nosubsidy": x.Annual_Income_USD / 1e5 * (1 - x.Subsidy_Available),
    })
    for level in range(1, 6):
        out[f"env{level}"] = (x.Environmental_Concern_Level == level).astype("float32")
        out[f"env{level}_subsidy"] = out[f"env{level}"] * x.Subsidy_Available
    for level in range(3):
        out[f"anx{level}"] = (x.Range_Anxiety_Level == level).astype("float32")
        out[f"anx{level}_subsidy"] = out[f"anx{level}"] * x.Subsidy_Available
    for col in [c for c in x.columns if c.startswith(("te_", "kernel_"))]:
        out[col] = to_logit(x[col].to_numpy(dtype="float64"))
    for col in ["recipe", "te_income_key", "te_income_10"]:
        for other in ["subsidy", "home", "env5", "anx0"]:
            out[f"{col}_x_{other}"] = out[col] * out[other]
    return out


def fit_linear(f, seed):
    train_x = linear_matrix(f["X_train"])
    scaler = StandardScaler().fit(train_x)
    model = LogisticRegression(C=2.0, max_iter=2000, solver="newton-cholesky")
    model.fit(scaler.transform(train_x), f["y_train"])
    return tuple(model.decision_function(scaler.transform(linear_matrix(f[k]))) for k in ("X_valid", "X_test"))


start = time.time()
oof_lin, test_lin, scores_lin, bag_lin = run_cv("Logistic", fit_linear)
print(f"Logistic elapsed: {time.time() - start:.0f} seconds")

**Conclusion.** The logistic regression reaches 0.94533, 0.0008 below the tree models, in only 38 seconds. It is weaker on its own but a different kind of model, and notebook 3 checks whether that difference helps the stack.

## Feature importance

Train one small LightGBM (400 trees) on the first fold and show which features it uses most, measured by total gain. This checks that the model relies on sensible columns. The ten largest gains are printed.

In [ ]:
gain = lgb.LGBMClassifier(**{**LGB_PARAMS, "n_estimators": 400}).fit(
    folds[0]["X_train"], folds[0]["y_train"], init_score=folds[0]["m_train"])
importance = pd.Series(gain.booster_.feature_importance("gain"), index=features).sort_values()
print(importance.tail(10)[::-1].round(1).to_string())
importance.tail(20).plot.barh(figsize=(8, 7), title="LightGBM gain importance")
plt.savefig(FIGURES / "06_feature_importance.png", dpi=110, bbox_inches="tight")
plt.show()

**Conclusion.** The four largest gains belong to `te_income_key` (92,910), `te_income_10` (54,346), `Environmental_Concern_Level` (41,071) and `recipe_score` (38,126). The encoded exact income carries far more gain than raw income (25,832), so the purchase rate of each exact income value adds information beyond income itself. Environmental concern and the recipe score match notebook 1, where concern, subsidy and income had the strongest links with the target; `Subsidy_Available` itself ranks low because the recipe score already contains it.

## Save the predictions

Save the out-of-fold and test predictions of the three models in float64 together with the row ids, so notebook 3 gets exactly the arrays computed here and can check that its rows line up. The tuned parameters go to `lgb_params.json`. The cell prints a SHA-256 fingerprint of every array, which makes two runs easy to compare.

In [ ]:
arrays = {
    "train_id": train["id"].to_numpy(),
    "test_id": test["id"].to_numpy(),
    "oof_lgb": oof_lgb, "test_lgb": test_lgb,
    "oof_xgb": oof_xgb, "test_xgb": test_xgb,
    "oof_lin": oof_lin, "test_lin": test_lin,
}
np.savez("own_predictions.npz", **arrays)
Path("lgb_params.json").write_text(json.dumps({"best_params": study.best_params, "best_fold0_auc": study.best_value,
                                               "trials_completed": len(study.trials)}, indent=2), encoding="utf-8")
for name, values in arrays.items():
    digest = hashlib.sha256(np.ascontiguousarray(values).tobytes()).hexdigest()[:16]
    print(f"{name}: {values.dtype} {values.shape} sha256 {digest}")
print(f"own_predictions.npz size: {Path('own_predictions.npz').stat().st_size / 1e6:.1f} MB")

**Conclusion.** `own_predictions.npz` holds eight arrays, 30.6 MB in all: the 668,665 train ids and 286,571 test ids as int64, and the out-of-fold and test predictions of the three models as float64. The fingerprints let a second run be checked against the first array by array.

## Summary

On 10 folds, LightGBM reaches an out-of-fold AUC of 0.94617, XGBoost 0.94616 and the logistic regression 0.94533, against 0.9377 for the recipe score alone. The target encoding of exact incomes is the largest feature by gain.

Notebook 3 reads `own_predictions.npz`, compares the three models and stacks them with public out-of-fold predictions.